In [37]:
from dotenv import load_dotenv
load_dotenv()

True

In [38]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = 'john@demo.com'
    password: str = 'password123'

In [39]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [40]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Juli,
    I'm going to be in town next week and was wondering if we could grab a coffee?
    Best,
    Jane
    """

@tool
def reply_email(to: str, subject: str, body: str) -> str:
    """Send a response to email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime[EmailContext]) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True,
            "messages": [ToolMessage(
                "Successfully authenticated",
                tool_call_id=runtime.tool_call_id
            )]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Wrong credentials",
                tool_call_id=runtime.tool_call_id
            )]
        })

In [41]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")

    if authenticated:
        tools = [check_inbox, reply_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools)
    return handler(request)

In [42]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = 'you are a helpful assistant that can check the inbox and reply emails.'
unauthenticated_prompt = 'you are a helpful assistant that can authenticate users.'

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get('authenticated')

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [43]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "gpt-5-nano",
    tools=[authenticate, check_inbox, reply_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call,
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                'authenticate': False,
                'check_inbox': False,
                'reply_email': True
            }
        )
    ]
)

In [44]:
from langchain.messages import HumanMessage

config = {'configurable': {'thread_id': '1'}}

response = agent.invoke(
    {'messages': [HumanMessage(content='Please check my inbox')]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

I can help with that, but I need to sign you in first. Please provide:

- Email:
- Password:

I’ll use these only to authenticate this session and won’t store your credentials. After sign-in, I can help you view your inbox.

If you’d rather not share your password here, I can guide you through checking your inbox directly in your webmail or email app instead.


In [45]:
from langchain.messages import HumanMessage

config = {'configurable': {'thread_id': '1'}}

response = agent.invoke(
    {'messages': [HumanMessage(content='john@demo.com, password123')]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

Here’s what I found in your inbox:

From Jane: "Hi Juli, I'm going to be in town next week and was wondering if we could grab a coffee? Best, Jane"

Would you like me to reply? I can draft a couple of options. Also, please confirm the recipient email address to send to (the address isn’t shown in this view). If you want, I can reply to this thread without needing the exact address by using the sender info in the message.

Draft options (you can pick one or tell me to customize):
- Option A (casual):
  - Subject: Re: Coffee next week
  - Body: Hi Jane, that sounds great! I’m around next week. What day works for you? Looking forward to catching up. Best, John

- Option B (slightly more specific):
  - Subject: Re: Coffee next week
  - Body: Hi Jane, I’d love to. I’m free Tuesday or Thursday afternoon next week—do one of those times work for you? Best, John

Please tell me:
1) Which draft you want (A, B, or a custom message)
2) The recipient email address to send to (or confirm I should re

In [46]:
from langchain.messages import HumanMessage

config = {'configurable': {'thread_id': '1'}}

response = agent.invoke(
    {'messages': [HumanMessage(content='Option A')]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

Option A selected. Here’s the final email draft:

Subject: Re: Coffee next week
To: [Jane's email address]
Body:
Hi Jane, that sounds great! I’m around next week. What day works for you? Looking forward to catching up. Best, John

Please provide Jane’s email address to send to (or confirm I should reply to Jane using the address on this thread, if you prefer). If you’d like, I can also tailor the message further (tone, times, etc.).


In [47]:
from langchain.messages import HumanMessage

config = {'configurable': {'thread_id': '1'}}

response = agent.invoke(
    {'messages': [HumanMessage(content='jane@demo.com')]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)